# TWII


In [16]:
import os
import textwrap
import math
import pandas as pd
import numpy as np

from datetime import datetime
from dateutil.relativedelta import relativedelta

import google.generativeai as genai
from langchain.schema.messages import AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory

from IPython.display import display
from IPython.display import Markdown

model = 'gemini-1.5-flash'
temperature = 1
top_p = 0.9
seed = 0

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

llm = ChatGoogleGenerativeAI(
                google_api_key = os.getenv('GEMINI_API_KEY'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
                
            })

path_price = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'tw', 'twii_tech.csv')
# path_news_title = os.path.dirname(os.path.abspath(os.getcwd())) + "/history_data/tw/news_title/" + id + "news_title.json"
path_out = "out_twii/compare_reverse/"

In [17]:
# 定義策略元素字典，每個元素對應一個代號
elements = {
    "E1": "企業創新策略：企業創新的核心主題，涵蓋所有驅動企業增長和競爭力提升的策略。",
    "E2": "滿足客戶需求：企業創新的首要驅動力，能否滿足客戶需求直接影響創新成敗。",
    "E3": "吸引投資者關注：投資者的關注和信心能為企業提供必要的資金和支持，是創新成功的重要標誌。",
    "E4": "市場需求與趨勢：市場需求的變化和行業趨勢是影響企業股票表現的關鍵因素，企業需適應並預測市場變化。",
    "E5": "財務表現：穩定且增長的收入、利潤和現金流是吸引投資者的主要因素，能增加投資者信心，推動股價上漲。",
    "E6": "技術進步與研發投入：持續的技術創新和研發投入能提高企業競爭力和市場地位，促進企業長期增長。",
    "E7": "經營效率：有效的經營管理能提高企業盈利能力，進而提升股票價值。",
    "E8": "品牌聲譽與市場認可度：良好的品牌聲譽和市場認可度能增加企業產品的市場需求，促進銷售增長。",
    "E9": "政策與法規支持：政府政策和法規對行業的支持或限制會直接影響企業的發展前景和市場表現。",
    "E10": "競爭優勢與市場份額：企業在市場中的競爭優勢和市場份額的提升能增強其市場地位，吸引更多投資者。",
    "E11": "全球化與國際市場拓展：進入國際市場並擴展全球業務能增加企業的市場規模和收入來源。",
    "E12": "風險管理與應對能力：有效的風險管理策略和應對能力能降低企業運營風險，提高投資者信心。",
    "E13": "人才管理與員工激勵：擁有高素質的人才和有效的激勵機制能提升企業創新能力和生產效率。",
    "E14": "合作夥伴關係：與其他企業建立戰略合作夥伴關係能共享資源和技術，提升市場競爭力。",
    "E15": "市場營銷與品牌推廣：有效的市場營銷和品牌推廣策略能提高企業知名度和市場需求。",
    "E16": "客戶服務與滿意度：高品質的客戶服務和較高的客戶滿意度能提高客戶忠誠度和口碑。",
    "E17": "供應鏈管理：高效的供應鏈管理能確保產品質量和交付效率，降低成本。",
    "E18": "產品多樣化與創新：不斷推出創新產品和服務能滿足不同市場需求，提升企業競爭力。",
    "E19": "環保與可持續發展：採用環保和可持續發展策略能提高企業形象，吸引關注可持續投資的投資者。",
    "E20": "數位轉型與科技應用：積極進行數位轉型和應用新技術能提高運營效率和市場響應速度。",
    "E21": "競爭對手分析與市場擴展：深入分析競爭對手的策略和行動，制定有效的市場擴展和新市場進入策略。",
    "E22": "資本結構、財務穩定性與資金籌措：健康的資本結構和財務穩定性以及有效的資金籌措能增強企業的抗風險能力和持續發展能力。",
    "E23": "客戶數據分析：利用大數據和數據分析技術深入了解客戶需求，提升產品和服務的針對性。",
    "E24": "知識產權保護：有效保護企業的知識產權能防止競爭對手抄襲，提高創新成果的市場價值。",
    "E25": "企業文化與價值觀：建立積極向上的企業文化和價值觀能吸引優秀人才，提升企業凝聚力和創新力。",
    "E26": "股東回報與分紅政策：穩定的股東回報和合理的分紅政策能吸引長期投資者。",
    "E27": "社會責任與公益活動：積極參與社會責任和公益活動能提升企業形象，獲得更多社會支持。",
    "E28": "成本控制: 有效的成本控制能提高企業獲利能力。",
}

# Prompts

p1

In [18]:
# 台指期的{prompt_tech}
# 權值股的表現往往會影響大盤的走勢，以下是台灣股市權值股的新聞標題：
# 『{text_news}』

# 請一步一步的用上面的資訊思考台指期隔天開盤買進到收盤賣出的獲利可能性，如果你認為會獲利請回答#yes，如果你認為會虧損請回答#no

# 用以下格式回答，你只需要綜合所有的新聞給出一次的回答就好：
# 判斷結果: #yes/#no
# 你的理由: 說明你的判斷理由


p2

In [19]:
# 透過前一天的新聞判斷今日的股市走勢
import time
import os
import json
import pandas as pd
from datetime import datetime, timedelta

def call_tech(d):
    df = pd.read_csv(path_price, encoding='utf-8')
    df['日期'] = pd.to_datetime(df['日期'], format='%Y%m%d')
    df = df[(d == df['日期'])]

    # 計算5日均線
    ema5 = "上漲" if df["收盤指數"].values[0] > df['EMA5'].values[0] else "下跌"

    elements = [
        # f"收盤後股價在5日均線{ma5},10日均線{ma10},20日均線{ma20}",
        # f"RSI值為{round(df['RSI'].values[0], 3)}",
        # f"MACD指標值, DIF-MACD值:{dif_macd}",
        # f"KD值為,k:{round(df['K'].values[0],1)},d:{round(df['D'].values[0], 1)}",
        # f'開盤{df["開盤指數"].values[0]}，最高{df["最高指數"].values[0]}，最低{df["最低指數"].values[0]}，收盤{df["收盤指數"].values[0]}',
        # f"收盤後股價在12日加權平均線{ema12},26日加權平均線{ema26}",
        f"五日加權移動平均線為{ema5}趨勢",
    ]

    prompt = ""
    for i in range(len(elements)):
        prompt += f"{str(i+1)} : {elements[i]}\n"
    return prompt

def analyze_market(st, et, prompt):
    stock_ids = ['2308', '2317', '2330', '2382', '2412', '2454', '2881', '2882', '2891', '6505']

    # 讀取CSV檔案
    df_price_his = pd.read_csv(path_price, encoding='utf-8')
    df_price_his['日期'] = pd.to_datetime(df_price_his['日期'], format='%Y%m%d')
    df_price_his = df_price_his[(st <= df_price_his['日期']) & (df_price_his['日期'] <= et)]

    idx = 1
    batch_size = 500
    output_data = {}  # 使用字典來存儲每一天的輸出數據

    while idx < len(df_price_his):
        batch = []
        for i in range(batch_size):
            if idx + i >= len(df_price_his):
                break
            
            # 取得前一日日期和新聞標題
            news_date = df_price_his.iloc[idx + i]['日期'] - timedelta(days=1)
            text_news = ""
            for j, stock_id in enumerate(stock_ids):
                path_news_file = os.path.dirname(os.path.abspath(os.getcwd())) + "/history_data/tw/news_title/" + stock_id + "news_title.json"
                with open(path_news_file, 'r', encoding='utf-8') as f:
                    news_data = json.load(f)
                    text_news = "權值股的熱門新聞標題"
                    text_news += f"{news_data[news_date.strftime('%Y%m%d')]} \n"
            
            # 取得前一個交易日的技術指標
            last_date = df_price_his.iloc[idx + i - 1]['日期']
            prompt_tech = call_tech(last_date)
            # 當日股價的技術指標如下：
            # 『{tech_prompt}』

            # 設置初始消息列表，包括 SystemMessage 和第一個 HumanMessage
            messages = [
                ("system", "你是一個厲害的股市操盤手，專長是當日沖消"),
                ("human",
f"""
台指期的{prompt_tech}
權值股的表現往往會影響大盤的走勢，以下是台灣股市權值股的新聞標題：
『{text_news}』

請一步一步的用上面的資訊思考台指期隔天開盤買進到收盤賣出的獲利可能性，如果你認為會獲利請回答#yes，如果你認為會虧損請回答#no

用以下格式回答，你只需要綜合所有的新聞給出一次的回答就好：
判斷結果: #yes/#no
你的理由: 說明你的判斷理由
""")
            ]
            batch.append(messages)

        res = llm.batch(batch)

        for i in range(len(res)):
            # 被預測的日期
            date_str = df_price_his.iloc[idx + i]['日期'].strftime('%Y%m%d')
            output_data[date_str] = {
                "token": res[i].usage_metadata,
                "raw_data": res[i].content,
            }

        # time.sleep(60)
        idx += batch_size

    # 將所有輸出數據寫入到同一個json檔案中
    data = {
        "model": model,
        "temperature": temperature,
        "top_p": top_p,
        "seed": seed,
        "prompt": prompt,
        "output_data": output_data,
    }
    
    return data

In [20]:
def save_data(folder_path, data):
    # count = 1
    # while True:
    #     if os.path.exists(folder_path + str(count) + ".json"):
    #         count += 1
    #         continue
    #     else:
    #         out_file = folder_path + str(count) + ".json"
    #         break
    out_file = folder_path + ".json"
    json_content = json.dumps(data, ensure_ascii=False, indent=4)
    os.makedirs(os.path.dirname(out_file), exist_ok=True)
    with open(out_file, "w", encoding="UTF-8") as f:
        f.write(json_content)
    
    print(f"已儲存至{out_file}")
    return out_file

評分

In [21]:
import pandas as pd
import json
import os

def eval(output_data):
    # 計算準確率和平均報酬率
    gain = []
    loss = []
    gain_precision = []
    loss_precision = []

    tp = 0
    fp = 0
    tn = 0
    fn = 0

    token_input = 0
    token_output = 0

    path_price_his = os.path.dirname(os.path.abspath(os.getcwd())) + "/history_data/tw/twii_history.csv"
    df_price_his = pd.read_csv(path_price_his, encoding='utf-8')
    df_price_his['日期'] = pd.to_datetime(df_price_his['日期'], format='%Y%m%d')

    for date_str, result in output_data.items():
        token_input += result["token"]["input_tokens"]
        token_output += result["token"]["output_tokens"]
        sig_str = result["raw_data"].split("你的理由")[0]
        sig = 0

        if "unknown" in sig_str:
            sig = 0
        elif "yes" in sig_str:
            sig = 1
        elif "no" in sig_str:
            sig = -1
        else:
            sig = 0

        df_price = df_price_his[df_price_his['日期'] == pd.to_datetime(date_str, format='%Y%m%d')]

        if df_price.empty:
            continue

        rtn = (df_price.iloc[0]['收盤指數'] - df_price.iloc[0]['開盤指數']) / df_price.iloc[0]['開盤指數']

        if pd.isna(rtn):
            rtn = 0

        if sig > 0:
            if rtn > 0:
                tp += 1
                gain.append(rtn)
                gain_precision.append(rtn)
            else:
                fp += 1
                loss.append(rtn)
                loss_precision.append(rtn)
        elif sig == -1:
            if rtn < 0:
                tn += 1
                gain.append(-rtn)
            else:
                fn += 1
                loss.append(-rtn)

    accuracy = 0
    precision = 0
    recall = 0
    ev = 0
    if tp + fp + tn + fn == 0 or tp + fp == 0 or tp + fn == 0:
        accuracy = 0
        precision = 0
        recall = 0
        ev = 0
        precision_ev = 0
    elif len(gain) == 0 or len(loss) == 0:
        accuracy = (tp + tn) / (tp + fp + tn + fn)
        precision = tp / (tp + fp)
        recall = tp / (tp + fn)
        ev = 0
        if len(gain_precision) == 0 or len(loss_precision) == 0:
            precision_ev = 0
        else:
            precision_ev = (sum(gain_precision) / len(gain_precision))*precision + (sum(loss_precision) / len(loss_precision))*(1-precision)

    else:
        accuracy = (tp + tn) / (tp + fp + tn + fn)
        precision = tp / (tp + fp)
        recall = tp / (tp + fn)
        ev = (sum(gain) / len(gain))*accuracy + (sum(loss) / len(loss))*(1-accuracy)
        if len(gain_precision) == 0 or len(loss_precision) == 0:
            precision_ev = 0
        else:
            precision_ev = (sum(gain_precision) / len(gain_precision))*precision + (sum(loss_precision) / len(loss_precision))*(1-precision)

    result = {
        'tp' : tp,
        'fp' : fp,
        'tn' : tn,
        'fn' : fn,
        'avail_count': tp + fp + tn + fn,
        'ask_count': len(output_data),
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'ev': ev,
        'precision_ev' : precision_ev,
        'token_input': token_input,
        'token_output': token_output,
    }

    return result

# RUN

In [22]:
start_day = datetime(2023, 1, 1)
end_day = datetime(2024, 6, 30)

prompt = ""
dic_data = analyze_market(start_day, end_day, prompt)
result = eval(dic_data["output_data"])
dic_data["result"] = result
save_data(path_out + "ema_20230101_202406302", dic_data)


print(f"time period: {start_day} ~ {end_day}")
print(f'''
tp : {result["tp"]}, fp : {result["fp"]}, tn : {result["tn"]}, fn : {result["fn"]}, ask_count : {result["ask_count"]}
accuracy : {round(result["accuracy"],3) }, ev : {round(result["ev"], 7)},
precision : {round(result["precision"],5)}, precision_ev : {round(result["precision_ev"], 7)},
recall : {round(result["recall"],5)}''') 

print("_"*50) 

已儲存至out_twii/compare_reverse/ema_20230101_202406302.json
time period: 2023-01-01 00:00:00 ~ 2024-06-30 00:00:00

tp : 69, fp : 40, tn : 94, fn : 128, ask_count : 355
accuracy : 0.492, ev : 0.0002778,
precision : 0.63303, precision_ev : 0.0013685,
recall : 0.35025
__________________________________________________


# DEMO

## 結果整理

In [23]:
import json

with open(path_out + "ema_20230101_202406302.json" , "r", encoding='utf-8') as f:
    data = json.load(f)
    print(data["result"])

with open(path_out + "ema_r_20230101_202406302.json" , "r", encoding='utf-8') as f:
    data = json.load(f)
    print(data["result"])


{'tp': 69, 'fp': 40, 'tn': 94, 'fn': 128, 'avail_count': 331, 'ask_count': 355, 'accuracy': 0.49244712990936557, 'precision': 0.6330275229357798, 'recall': 0.350253807106599, 'ev': 0.0002778096347077477, 'precision_ev': 0.001368466597665854, 'token_input': 96467, 'token_output': 85253}
{'tp': 36, 'fp': 18, 'tn': 113, 'fn': 164, 'avail_count': 331, 'ask_count': 355, 'accuracy': 0.4501510574018127, 'precision': 0.6666666666666666, 'recall': 0.18, 'ev': -2.820691387388696e-05, 'precision_ev': 0.0023663250377801625, 'token_input': 96467, 'token_output': 82274}


## 作圖
使用 Best so far

In [24]:
import json

avail_count = 0
ask_count = 0

print("正確版本")
with open(path_out + "ema_20230101_20240630.json" , "r", encoding='utf-8') as f:
    data = json.load(f)
    result = data["result"]
    print(f'''
    tp : {result["tp"]}, fp : {result["fp"]}, tn : {result["tn"]}, fn : {result["fn"]}, ask_count : {result["ask_count"]}
    accuracy : {round(result["accuracy"],3) }, ev : {round(result["ev"], 7)},
    precision : {round(result["precision"],5)}, precision_ev : {round(result["precision_ev"], 7)},
    recall : {round(result["recall"],5)}''') 
    avail_count += data["result"]["avail_count"]
    ask_count += data["result"]["ask_count"]
print("_"*50)
print("錯誤版本")
with open(path_out + "ema_r_20230101_20240630.json" , "r", encoding='utf-8') as f:
    data = json.load(f)
    result = data["result"]
    print(f'''
    tp : {result["tp"]}, fp : {result["fp"]}, tn : {result["tn"]}, fn : {result["fn"]}, ask_count : {result["ask_count"]}
    accuracy : {round(result["accuracy"],3) }, ev : {round(result["ev"], 7)},
    precision : {round(result["precision"],5)}, precision_ev : {round(result["precision_ev"], 7)},
    recall : {round(result["recall"],5)}''') 
    avail_count += data["result"]["avail_count"]
    ask_count += data["result"]["ask_count"]

print(f"ask_count[{ask_count}/{avail_count}]")


正確版本

    tp : 64, fp : 40, tn : 95, fn : 132, ask_count : 355
    accuracy : 0.48, ev : -1.01e-05,
    precision : 0.61538, precision_ev : 0.0009337,
    recall : 0.32653
__________________________________________________
錯誤版本

    tp : 32, fp : 19, tn : 112, fn : 165, ask_count : 355
    accuracy : 0.439, ev : -0.0001157,
    precision : 0.62745, precision_ev : 0.0021432,
    recall : 0.16244
ask_count[710/659]
